In [6]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, fbeta_score

hu = pd.read_csv("data/processed/humaid_safety.csv")
t26 = pd.read_csv("data/processed/t26_safety.csv")
t26 = t26[t26["event"] != "2013_Sardinia_floods"]

HU_HOLD = ["maryland_floods_2018", "srilanka_floods_2017"]
T26_TRAIN = ["2013_Alberta_floods", "2013_Colorado_floods"]

hu_train = hu[(~hu["event"].isin(HU_HOLD)) & (hu["split"] == "train")][["text", "safety_label"]]
hu_dev = hu[(~hu["event"].isin(HU_HOLD)) & (hu["split"] == "dev")][["text", "safety_label"]]
t26_train = t26[t26["event"].isin(T26_TRAIN)][["text", "safety_label"]]
t_tr, t_dev = train_test_split(t26_train, test_size=0.2, random_state=0, stratify=t26_train["safety_label"])

train = pd.concat([hu_train, t_tr])
dev = pd.concat([hu_dev, t_dev])
tests = {
    "HumAID unseen floods":   hu[hu["event"].isin(HU_HOLD)],
    "T26 Queensland":         t26[t26["event"] == "2013_Queensland_floods"],
    "T26 Philippines+Manila": t26[t26["event"].isin(["2012_Philipinnes_floods", "2013_Manila_floods"])],
}
print(len(train), len(dev))     # should print 18080 2749, same as notebook 09

18080 2749


In [7]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

os.makedirs("data/processed/emb", exist_ok=True)

def embed(texts, cache):
    if os.path.exists(cache):
        return np.load(cache)
    E = model.encode(list(texts), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(cache, E)
    return E

X_train = embed(train["text"], "data/processed/emb/train.npy")
X_dev = embed(dev["text"], "data/processed/emb/dev.npy")
X_tests = [embed(d["text"], f"data/processed/emb/test_{i}.npy") for i, d in enumerate(tests.values())]
print(X_train.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\arbaz\Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arbaz\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (11 outputs left out)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

(18080, 384)


In [8]:
def best_threshold(p, y):
    best = (0.5, -1)
    for th in np.arange(0.20, 0.81, 0.05):
        f2 = fbeta_score(y, (p >= th).astype(int), beta=2, zero_division=0)
        if f2 >= best[1]:
            best = (th, f2)
    return best

def run(name, Xtr, Xdv, Xts):
    clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, train["safety_label"])
    hz = list(clf.classes_).index("hazard")
    y_dev = (dev["safety_label"] == "hazard").astype(int).values
    th, f2 = best_threshold(clf.predict_proba(Xdv)[:, hz], y_dev)
    rows = []
    for (tn, d), Xt in zip(tests.items(), Xts):
        y = (d["safety_label"] == "hazard").astype(int).values
        p = clf.predict_proba(Xt)[:, hz]
        pred = (p >= th).astype(int)
        tp = int((pred & y).sum())
        rows.append({"model": name, "threshold": round(th, 2), "dev_F2": round(f2, 2), "test": tn,
                     "precision": round(tp / max(int(pred.sum()), 1), 2),
                     "recall": round(tp / int(y.sum()), 2),
                     "avg_precision": round(average_precision_score(y, p), 3)})
    return rows

tf = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
Xtr_tf = tf.fit_transform(train["text"])
rows = run("TF-IDF", Xtr_tf, tf.transform(dev["text"]), [tf.transform(d["text"]) for d in tests.values()])
rows += run("MiniLM embeddings", X_train, X_dev, X_tests)
pd.DataFrame(rows)

,model,threshold,dev_F2,test,precision,recall,avg_precision
0,TF-IDF,0.4,0.87,HumAID unseen floods,0.49,0.80,0.737
1,TF-IDF,0.4,0.87,T26 Queensland,0.47,0.55,0.540
2,TF-IDF,0.4,0.87,T26 Philippines+Manila,0.38,0.17,0.253
3,MiniLM embeddings,0.4,0.84,HumAID unseen floods,0.44,0.88,0.697
4,MiniLM embeddings,0.4,0.84,T26 Queensland,0.45,0.72,0.567
5,MiniLM embeddings,0.4,0.84,T26 Philippines+Manila,0.35,0.34,0.320


In [9]:
tests7 = [
    "water is rushing down 5th avenue, cars are stuck up to the doors",
    "huge tree fell across the road on Queens Blvd, nobody can pass",
    "my neighbor is playing loud music again",
    "basement flooded with sewage after the storm",
    "the street is a river near the subway entrance",
    "nice weather today, going for a walk",
    "big branch came down on Atlantic Ave and blocks the lane",
]
clf_tf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr_tf, train["safety_label"])
clf_em = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_train, train["safety_label"])
hz_tf = list(clf_tf.classes_).index("hazard")
hz_em = list(clf_em.classes_).index("hazard")

p_tf = clf_tf.predict_proba(tf.transform(tests7))[:, hz_tf]
p_em = clf_em.predict_proba(model.encode(tests7, normalize_embeddings=True))[:, hz_em]
for t, a, b in zip(tests7, p_tf, p_em):
    print(f"TF-IDF {a:.2f} | MiniLM {b:.2f} | threshold 0.40 | {t}")

TF-IDF 0.30 | MiniLM 0.45 | threshold 0.40 | water is rushing down 5th avenue, cars are stuck up to the doors
TF-IDF 0.58 | MiniLM 0.98 | threshold 0.40 | huge tree fell across the road on Queens Blvd, nobody can pass
TF-IDF 0.21 | MiniLM 0.20 | threshold 0.40 | my neighbor is playing loud music again
TF-IDF 0.44 | MiniLM 0.89 | threshold 0.40 | basement flooded with sewage after the storm
TF-IDF 0.57 | MiniLM 0.22 | threshold 0.40 | the street is a river near the subway entrance
TF-IDF 0.11 | MiniLM 0.01 | threshold 0.40 | nice weather today, going for a walk
TF-IDF 0.41 | MiniLM 0.96 | threshold 0.40 | big branch came down on Atlantic Ave and blocks the lane


In [10]:
from scipy.sparse import hstack, csr_matrix

def comb(Xs, E):
    return hstack([Xs, csr_matrix(E)]).tocsr()

rows2 = run("TF-IDF + MiniLM combined",
            comb(Xtr_tf, X_train),
            comb(tf.transform(dev["text"]), X_dev),
            [comb(tf.transform(d["text"]), E) for d, E in zip(tests.values(), X_tests)])
pd.DataFrame(rows + rows2)

,model,threshold,dev_F2,test,precision,recall,avg_precision
0,TF-IDF,0.40,0.87,HumAID unseen floods,0.49,0.80,0.737
1,TF-IDF,0.40,0.87,T26 Queensland,0.47,0.55,0.540
2,TF-IDF,0.40,0.87,T26 Philippines+Manila,0.38,0.17,0.253
3,MiniLM embeddings,0.40,0.84,HumAID unseen floods,0.44,0.88,0.697
4,MiniLM embeddings,0.40,0.84,T26 Queensland,0.45,0.72,0.567
5,MiniLM embeddings,0.40,0.84,T26 Philippines+Manila,0.35,0.34,0.320
6,TF-IDF + MiniLM combined,0.45,0.87,HumAID unseen floods,0.58,0.82,0.750
7,TF-IDF + MiniLM combined,0.45,0.87,T26 Queensland,0.59,0.59,0.601
8,TF-IDF + MiniLM combined,0.45,0.87,T26 Philippines+Manila,0.54,0.17,0.347


In [11]:
from sklearn.metrics import precision_recall_curve

def prec_at_recall(y, p, target):
    pr, rc, _ = precision_recall_curve(y, p)
    ok = pr[rc >= target]
    return round(float(ok.max()), 2) if len(ok) else 0.0

def probs(Xtr, Xts):
    clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, train["safety_label"])
    hz = list(clf.classes_).index("hazard")
    return [clf.predict_proba(Xt)[:, hz] for Xt in Xts]

variants = {
    "TF-IDF": (Xtr_tf, [tf.transform(d["text"]) for d in tests.values()]),
    "MiniLM": (X_train, X_tests),
    "TF-IDF + MiniLM": (comb(Xtr_tf, X_train),
                        [comb(tf.transform(d["text"]), E) for d, E in zip(tests.values(), X_tests)]),
}
rows3 = []
for name, (Xtr, Xts) in variants.items():
    for (tn, d), p in zip(tests.items(), probs(Xtr, Xts)):
        y = (d["safety_label"] == "hazard").astype(int).values
        rows3.append({"model": name, "test": tn,
                      "precision_at_60%_recall": prec_at_recall(y, p, 0.60),
                      "precision_at_80%_recall": prec_at_recall(y, p, 0.80)})
pd.DataFrame(rows3)

,model,test,precision_at_60%_recall,precision_at_80%_recall
0,TF-IDF,HumAID unseen floods,0.71,0.51
1,TF-IDF,T26 Queensland,0.48,0.34
2,TF-IDF,T26 Philippines+Manila,0.13,0.12
3,MiniLM,HumAID unseen floods,0.72,0.57
4,MiniLM,T26 Queensland,0.50,0.39
5,MiniLM,T26 Philippines+Manila,0.29,0.23
6,TF-IDF + MiniLM,HumAID unseen floods,0.76,0.63
7,TF-IDF + MiniLM,T26 Queensland,0.57,0.42
8,TF-IDF + MiniLM,T26 Philippines+Manila,0.28,0.22


## Findings

- Same train (18,080), dev (2,749) and unseen test sets as notebook 09; TF-IDF row reproduces notebook 09 exactly. MiniLM (all-MiniLM-L6-v2) used as frozen embeddings + logistic regression; threshold chosen on dev by F2 (0.40 for both).
- Average precision, TF-IDF to MiniLM: HumAID unseen floods 0.737 to 0.697, T26 Queensland 0.540 to 0.567, T26 Philippines+Manila 0.253 to 0.320. Recall: 0.80 to 0.88, 0.55 to 0.72, 0.17 to 0.34. Dev F2: 0.87 vs 0.84.
- Mixed result: embeddings transfer slightly better to T26 but are slightly worse on held-out HumAID floods; differences of 0.03–0.04 are within noise (86–119 hazard tweets per test, single run). Philippines+Manila remains weak (recall 0.34).
- Interpretation (not proven): meaning helps a little across datasets, but label disagreement between datasets is not fixed by better text features.
- Role: second baseline between TF-IDF (baseline v2) and the fine-tuned Qwen 2.5 (QLoRA).
- Not tested: fine-tuned encoder, larger embedding models.
- 7 hand-written sentences at threshold 0.40: TF-IDF and MiniLM each caught 4 of 5 hazard sentences with no false alarm, but missed different ones (TF-IDF missed "water is rushing down 5th avenue"; MiniLM missed the metaphor "the street is a river"). Illustration only, not a score.
- TF-IDF + MiniLM combined (threshold 0.45 chosen on dev, dev F2 0.87): best average precision on all three unseen tests (0.750, 0.601, 0.347 vs TF-IDF 0.737, 0.540, 0.253). Recall on Philippines+Manila stays 0.17 at the dev threshold.
- Selection caveat: three models compared on the same three small test sets, single run; gains of 0.01–0.09 are suggestive, not conclusive.
- Role: baseline v2 = TF-IDF (notebook 09); strongest classical variant = TF-IDF + MiniLM; target for the fine-tuned Qwen 2.5 (QLoRA).